# 🏥 Medical Question Answering - Full Pipeline

**Projet Deep Learning - Clé en main**

Ce notebook exécute automatiquement:
1. Installation des dépendances
2. Téléchargement du dataset
3. Expériences (Flan-T5, OpenAI si clé fournie)
4. Évaluation (ROUGE, BLEU)
5. Génération du rapport PDF

⚠️ **Medical Disclaimer**: Projet éducatif uniquement. Ne pas utiliser pour des conseils médicaux.

## 1️⃣ Configuration (EXÉCUTER EN PREMIER)

In [ ]:
#@title ⚙️ Configuration
#@markdown ### Clés API (optionnel mais recommandé)
OPENAI_API_KEY = ""  #@param {type:"string"}
#@markdown ### Mode d'exécution
MODE = "quick"  #@param ["quick", "full"]
#@markdown - **quick**: 50 exemples, ~10 min
#@markdown - **full**: 500 exemples, ~1h

import os
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✅ Clé OpenAI configurée")
else:
    print("⚠️ Pas de clé OpenAI - seules les expériences locales seront exécutées")

QUICK_MODE = MODE == "quick"
print(f"Mode: {MODE}")

## 2️⃣ Installation

In [ ]:
#@title 📦 Cloner le repo et installer les dépendances
%%time

import os

# Clone repo
if not os.path.exists("Deep-Learning"):
    !git clone https://github.com/salma-zr/Deep-Learning.git
    %cd Deep-Learning
    !git checkout cursor/compl-tude-de-t-che-8adb
else:
    %cd Deep-Learning
    !git pull origin cursor/compl-tude-de-t-che-8adb

# Install dependencies
!pip install -q -e .

print("\n✅ Installation terminée!")

## 3️⃣ Préparation des données

In [ ]:
#@title 📊 Télécharger le dataset et créer les splits
%%time

!python -m src.data.cli download
!python -m src.data.cli split --seed 42 --tiny-size 200
!python -m src.data.cli info

print("\n✅ Dataset prêt!")

## 4️⃣ Expériences

In [ ]:
#@title 🧪 Créer les dossiers de résultats
!mkdir -p results/preds results/scores results/qualitative results/figures
!mkdir -p report/tables report/figures

In [ ]:
#@title 🤖 Expérience 1: Flan-T5 (CPU, gratuit)
%%time

split = "tiny_test" if QUICK_MODE else "test"
limit = 50 if QUICK_MODE else 500

!python -m src.generation.cli run configs/exp_11_hf_flan_t5.yaml --split {split} --limit {limit}
print("\n✅ Expérience Flan-T5 terminée!")

In [ ]:
#@title 🤖 Expérience 2: Flan-T5 + Flashcard Prompt
%%time

!python -m src.generation.cli run configs/exp_12_hf_flashcard.yaml --split {split} --limit {limit}
print("\n✅ Expérience Flashcard terminée!")

In [ ]:
#@title 🤖 Expérience 3: Flan-T5 + One-Sentence Prompt
%%time

!python -m src.generation.cli run configs/exp_13_hf_one_sentence.yaml --split {split} --limit {limit}
print("\n✅ Expérience One-Sentence terminée!")

In [ ]:
#@title 🌐 Expérience 4: OpenAI GPT-4o-mini (nécessite clé API)
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.generation.cli run configs/exp_01_openai_baseline.yaml --split {split} --limit {limit}
    print("\n✅ Expérience OpenAI terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

In [ ]:
#@title 🌐 Expérience 5: OpenAI + Flashcard Prompt
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.generation.cli run configs/exp_04_prompt_flashcard.yaml --split {split} --limit {limit}
    print("\n✅ Expérience OpenAI Flashcard terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

In [ ]:
#@title 🌐 Expérience 6: OpenAI + Uncertainty Prompt
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.generation.cli run configs/exp_05_prompt_uncertainty.yaml --split {split} --limit {limit}
    print("\n✅ Expérience OpenAI Uncertainty terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

In [ ]:
#@title 📚 Expérience 7: RAG Wikipedia (nécessite clé API)
%%time

import os
if os.environ.get("OPENAI_API_KEY"):
    !python -m src.rag.cli run configs/exp_06_rag_wikipedia.yaml --split {split} --limit {limit}
    print("\n✅ Expérience RAG Wikipedia terminée!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI")

## 5️⃣ Évaluation

In [ ]:
#@title 📈 Calculer les métriques pour toutes les expériences
%%time

import glob

pred_files = glob.glob("results/preds/*.jsonl")
print(f"Fichiers de prédictions trouvés: {len(pred_files)}")

for pred_file in pred_files:
    if "_judged" in pred_file:
        continue
    name = pred_file.split("/")[-1].replace(".jsonl", "")
    print(f"\n📊 Évaluation de {name}...")
    !python -m src.eval.cli metrics "{pred_file}" --output "results/scores/{name}.csv"

print("\n✅ Évaluation terminée!")

In [ ]:
#@title 🧑‍⚖️ LLM Judge (nécessite clé API)
%%time

import os
import glob

if os.environ.get("OPENAI_API_KEY"):
    pred_files = glob.glob("results/preds/*.jsonl")
    for pred_file in pred_files:
        if "_judged" in pred_file:
            continue
        name = pred_file.split("/")[-1].replace(".jsonl", "")
        print(f"\n🧑‍⚖️ Judging {name}...")
        !python -m src.eval.cli judge "{pred_file}" --judge-model gpt-4o-mini --limit 50
    print("\n✅ Jugement terminé!")
else:
    print("⏭️ Skipped - Pas de clé OpenAI pour le LLM Judge")

## 6️⃣ Génération du Rapport

In [ ]:
#@title 📊 Générer les tableaux et figures
%%time

!python -m src.report.cli tables
!python -m src.report.cli figures

print("\n✅ Tableaux et figures générés!")

In [ ]:
#@title 📄 Installer LaTeX et compiler le PDF
%%time

# Install LaTeX
!apt-get update -qq && apt-get install -qq -y texlive-latex-base texlive-latex-extra > /dev/null 2>&1

# Compile PDF
!python -m src.report.cli compile report/report.tex

print("\n✅ PDF compilé!")

In [ ]:
#@title 📥 Télécharger le rapport PDF
from google.colab import files

# Download PDF
files.download('report/report.pdf')
print("\n✅ Téléchargement lancé!")

## 7️⃣ Visualisation des Résultats

In [ ]:
#@title 📊 Afficher le tableau des résultats
import pandas as pd
import glob

# Load all score files
score_files = glob.glob("results/scores/*.csv")
if score_files:
    dfs = [pd.read_csv(f) for f in score_files]
    results = pd.concat(dfs, ignore_index=True)
    
    # Display key metrics
    cols = ['experiment', 'rougeL', 'bleu', 'latency_mean_ms', 'n_examples']
    cols = [c for c in cols if c in results.columns]
    display(results[cols].sort_values('rougeL', ascending=False))
else:
    print("Aucun fichier de scores trouvé")

In [ ]:
#@title 📈 Afficher les figures générées
from IPython.display import Image, display
import os

figures = [
    "report/figures/metrics_comparison.png",
    "report/figures/judge_distribution.png",
    "report/figures/ablation_curve.png"
]

for fig in figures:
    if os.path.exists(fig):
        print(f"\n📊 {fig}")
        display(Image(fig, width=600))

In [ ]:
#@title 🔍 Exemples de prédictions
import json
import glob

pred_files = glob.glob("results/preds/*.jsonl")
if pred_files:
    # Show examples from first file
    with open(pred_files[0]) as f:
        examples = [json.loads(line) for line in f][:5]
    
    print(f"📂 Fichier: {pred_files[0]}\n")
    for i, ex in enumerate(examples, 1):
        print(f"--- Exemple {i} ---")
        print(f"❓ Question: {ex['question'][:100]}...")
        print(f"✅ Référence: {ex['reference'][:100]}...")
        print(f"🤖 Prédiction: {ex['prediction']}")
        print()

## 8️⃣ Fine-tuning (Optionnel, GPU requis)

In [ ]:
#@title 🔧 Vérifier GPU disponible
import torch

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ Pas de GPU - Le fine-tuning sera très lent ou impossible")
    print("   → Va dans Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 📚 Préparer les données de fine-tuning
%%time

!python -m src.finetune.cli prepare --max 1000 --style alpaca
print("\n✅ Données préparées!")

In [ ]:
#@title 🚀 Lancer le fine-tuning (1k exemples)
%%time

import torch

if torch.cuda.is_available():
    !python -m src.finetune.cli train configs/exp_08_finetune_1k.yaml
    print("\n✅ Fine-tuning terminé!")
else:
    print("⏭️ Skipped - Pas de GPU disponible")
    print("   Exécution en mode symbolique pour test...")
    !python -m src.finetune.cli train configs/exp_08_finetune_1k.yaml --symbolic

## 📥 Télécharger tous les résultats

In [ ]:
#@title 📦 Créer une archive ZIP de tous les résultats
!zip -r medical_qa_results.zip results/ report/report.pdf report/tables/ report/figures/

from google.colab import files
files.download('medical_qa_results.zip')
print("\n✅ Archive téléchargée!")

---
## ✅ Checklist Finale

Avant de soumettre, vérifie que:

- [ ] Le PDF `report/report.pdf` est généré
- [ ] Les tableaux montrent des résultats réels
- [ ] Les figures sont lisibles
- [ ] Tu as personnalisé la section "Qualitative Analysis" dans le rapport
- [ ] Tu as relu la section "Discussion & Limitations"

**Bon courage !** 🎓